In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import sqlalchemy as db
import sys
from dotenv import load_dotenv
load_dotenv("../configuration/.env")
# Move up one level to the project_root and then access src
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from configuration.config import Config
from src.db_connection import connect_to_mysql
from src.data_loading import load_fact_watchs
from src.data_prosessing import preprocess_data
from src.data_clustering import train_model

from sklearn.cluster import KMeans
from ydata_profiling import ProfileReport

# Loading information from data warehouse

## Data Warehouse Connection

In [2]:
username = os.getenv('MYSQL_USER')
password = os.getenv('MYSQL_PASSWORD')
host = os.getenv('MYSQL_HOST')
port = os.getenv('MYSQL_PORT')
database = os.getenv('MYSQL_DATABASE_DW')
uri = f"mysql://{username}:{password}@{host}/{database}"
print(uri)

dw_engine = db.create_engine(uri)

mysql://root:@localhost/dw_netflix


## get data

In [18]:
query = """
SELECT factwatchs.userID,
        dimmovie.title, dimmovie.releaseDate, dimmovie.genre, dimmovie.award as movieAward,
        dimscore.netflixScore, dimscore.imdbScore, dimscore.rottentomatoesScore, dimscore.sensacineScore, dimscore.peopleScore,
        dimperformer.name as performerName, dimperformer.role performerRole, dimperformer.award as performerAward,
        diminteraction.finishCount, diminteraction.backClickCount, diminteraction.movieForwardCount, diminteraction.playCount, diminteraction.movieViewPercentage
    FROM factwatchs
    INNER JOIN dimmovie ON factwatchs.movieID = dimmovie.movieID
    INNER JOIN dimscore ON factwatchs.scoreID = dimscore.scoreID
    LEFT JOIN dimperformer ON factwatchs.performerID = dimperformer.performerID
    LEFT JOIN dimuser ON factwatchs.userID = dimuser.userID
    LEFT JOIN diminteraction ON factwatchs.interactionID = diminteraction.interactionID
"""

fact_watchs_df = pd.read_sql(query,dw_engine)

## prepocess data

In [19]:
fact_watchs_df["releaseDate"] = pd.to_datetime(fact_watchs_df["releaseDate"])

score_columns = ["netflixScore", "imdbScore", "rottentomatoesScore", "sensacineScore"]
people_score_mask_null = fact_watchs_df["peopleScore"].isnull()
fact_watchs_df.loc[~people_score_mask_null, "peopleScore"] = (fact_watchs_df.loc[~people_score_mask_null, "peopleScore"] / 2) * 0.1
fact_watchs_df.loc[people_score_mask_null, "peopleScore"] = fact_watchs_df.loc[people_score_mask_null, score_columns].mean(axis=1, skipna=True)

fact_watchs_df["movieAward"] = fact_watchs_df["movieAward"].fillna("Sin Info")
fact_watchs_df["performerName"].fillna("No Info")
fact_watchs_df["performerRole"].fillna("No Info")

print(fact_watchs_df.columns)
print("\n")
print(fact_watchs_df.info())

Index(['userID', 'title', 'releaseDate', 'genre', 'movieAward', 'netflixScore',
       'imdbScore', 'rottentomatoesScore', 'sensacineScore', 'peopleScore',
       'performerName', 'performerRole', 'performerAward', 'finishCount',
       'backClickCount', 'movieForwardCount', 'playCount',
       'movieViewPercentage'],
      dtype='object')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88991 entries, 0 to 88990
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   userID               88991 non-null  int64         
 1   title                88991 non-null  object        
 2   releaseDate          88914 non-null  datetime64[ns]
 3   genre                88991 non-null  object        
 4   movieAward           88991 non-null  object        
 5   netflixScore         86567 non-null  float64       
 6   imdbScore            86667 non-null  float64       
 7   rottentomatoesScore  86452 no

C:\Users\joa_g\AppData\Local\Temp\ipykernel_12796\2221047952.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  fact_watchs_df["performerName"].fillna("No Info", inplace=True)
C:\Users\joa_g\AppData\Local\Temp\ipykernel_12796\2221047952.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as 

### summarize score

In [20]:
score_columns = ["netflixScore", "imdbScore", "rottentomatoesScore", "sensacineScore","peopleScore"]
fact_watchs_df["score"] = fact_watchs_df[score_columns].mean(axis=1,skipna=True)
fact_watchs_df.drop(columns=score_columns,inplace=True)

# Analize

In [21]:
profile = ProfileReport(fact_watchs_df, title="movie_data Profiling Report")
profile.to_widgets()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\joa_g\AppData\Local\Programs\Python\Python312\Lib\site-packages\ydata_profiling\model\pandas\summary_pandas.py:39: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  series = series.fillna(np.nan)


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render widgets:   0%|          | 0/1 [00:00<?, ?it/s]

# Clustering 

## K-Means Clustering by user favourite genre

In [17]:
tabla_model = pd.pivot_table(fact_watchs_df, values='score', index=['userID'],
                       columns=['genre'], aggfunc="mean").reset_index()

tabla_model = tabla_model.fillna(0)

tabla_model.tail(10)

genre,userID,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
9699,9828,0.00,0.0,0.0,0.0,1.50,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9700,9829,0.00,0.0,0.0,0.0,0.00,0.0,0.0,1.75,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9701,9830,2.75,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9702,9831,3.25,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9703,9832,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.75,0.0,0.0,0.0
9704,9833,0.00,2.5,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9705,9834,0.00,0.0,0.0,0.0,2.25,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9706,9835,0.00,0.0,0.0,0.0,2.25,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9707,9836,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,2.25,0.0,0.0,0.0,0.00,0.0,0.0,0.0
9708,9837,0.00,0.0,0.0,0.0,0.00,0.0,0.0,2.75,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,0.0,0.0


In [ ]:
X1 = tabla_model[['Action', 'Adventure', 'Animation', 'Children', 'Comedy',
       'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western']].iloc[: , :].values

inertia = []
for n in range(1 , 11):
    algorithm = (KMeans(n_clusters = n ,init='k-means++', n_init = 20 ,max_iter=300,
                        tol=0.0001,  random_state= 111  , algorithm='elkan') )
    algorithm.fit(X1)
    inertia.append(algorithm.inertia_)

In [ ]:
plt.figure(1 , figsize = (15 ,6))
plt.plot(np.arange(1 , 11) , inertia , 'o')
plt.plot(np.arange(1 , 11) , inertia , '-' , alpha = 0.5)
plt.xlabel('Number of Clusters') , plt.ylabel('Inertia')
plt.show()

### select number of clusters to train a model

In [ ]:
#  use fit_predict to cluster the dataset
n_clusters = 6
predictions = train_model(X1,n_clusters)

tabla_model['Cluster'] = predictions

### now we can classify each user to its favorite genre and suggest movies

In [ ]:
movies_datacluster = fact_watchs_df.merge(tabla_model[['userID','Cluster']],on='userID',how='left')

movies_datacluster.head()

we can check to which cluster does the user #61 belong

In [ ]:
user_id = 61
user_filter = movies_datacluster["userID"]==user_id
user_cluster = movies_datacluster[user_filter]["Cluster"].head(1).values[0]

print(f"user {user_id} belongs to cluster {user_cluster}")

Top movies from user's cluster

In [ ]:
cluster_filter = movies_datacluster["Cluster"]==user_cluster
print(f"cluster {user_cluster} belongs to genre {movies_datacluster[cluster_filter]["genre"].head(1).values}")

ranking_cluster = movies_datacluster[cluster_filter].groupby(['title'])["score"].mean().sort_values(ascending=False)
ranking_cluster = ranking_cluster.reset_index()
ranking_cluster

what movies did the user watch?

In [ ]:
# user watched
movies_watched = movies_datacluster[user_filter]["title"]
print(f"user {user_id} watched \n {movies_watched}")

print("\n")

# user did not watch
movies_not_watched = ranking_cluster[~ranking_cluster["title"].isin(movies_watched)]
movies_not_watched.sort_values("score",ascending=False).head(10)
print(f"user {user_id} did not watch \n {movies_not_watched}")

## K-Means Clustering by user favourite genre inferred by the movies he watched

In [ ]:
user_movie_ratings=pd.pivot_table(fact_watchs_df, values='score', index=['userID'],
                       columns=['title'], aggfunc="mean")

user_movie_ratings

we will generate clusters based on the movies the users have seen and scored

In [ ]:
from scipy.sparse import csr_matrix

for column in user_movie_ratings.columns:
    user_movie_ratings[column] = pd.arrays.SparseArray(user_movie_ratings[column], dtype=pd.SparseDtype("float", 0))

user_movie_ratings = user_movie_ratings.fillna(0)

user_movie_ratings.head()

In [ ]:
sparse_ratings = csr_matrix(user_movie_ratings.sparse.to_coo())

In [ ]:
predictions = KMeans(n_clusters=20, algorithm='lloyd').fit_predict(sparse_ratings)

In [ ]:
clustered = pd.concat([user_movie_ratings.reset_index(), pd.DataFrame({'group':predictions})], axis=1)

movies_datacluster_peliculas = fact_watchs_df.merge(clustered[['userID','group']],on='userID',how='left')

movies_datacluster_peliculas

let's check user #61 again what movies has he watched

In [ ]:
user_filter = movies_datacluster_peliculas["userID"]==61
user_61_cluster = movies_datacluster_peliculas[user_filter]["group"].head(1).values[0]
movies_datacluster_peliculas[user_filter][["group","genre"]]


movies on users cluster

In [ ]:
top_peliculas = clustered[clustered["group"]==user_61_cluster][clustered.columns[1: len(clustered.columns) -1]].mean().reset_index()
top_peliculas.rename(columns={0:"ranking"},inplace=True)
top_peliculas

what movies can we recommend to user #61

In [ ]:
top_peliculas[ ~top_peliculas["title"].isin(movies_datacluster_peliculas[movies_datacluster_peliculas ["userID"]==61]["title"])].sort_values("ranking",ascending=False)[0:10]